# Review Enrichment (Step-by-Step)

This notebook joins raw reviews with user + business metadata and writes an enriched review document to MongoDB.

Assumptions:
- MongoDB is running and accessible from this notebook container.
- Spark has the MongoDB Spark Connector available (per project images).
- Collections: `reviews`, `users`, `businesses`.


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, struct

# Mongo config (adjust if needed)
MONGO_HOST = os.getenv("MONGO_HOST", "mongodb")
MONGO_PORT = os.getenv("MONGO_PORT", "27017")
MONGO_USER = os.getenv("MONGO_USER", os.getenv("MONGO_INITDB_ROOT_USERNAME", "root"))
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD", os.getenv("MONGO_INITDB_ROOT_PASSWORD", "password"))
MONGO_AUTH_DB = os.getenv("MONGO_AUTH_DB", "admin")
MONGO_DB = os.getenv("MONGO_DB", "yelp")

mongo_uri = (
    f"mongodb://{MONGO_USER}:{MONGO_PASSWORD}@{MONGO_HOST}:{MONGO_PORT}/"
    f"{MONGO_AUTH_DB}?authSource={MONGO_AUTH_DB}"
)

spark = (
    SparkSession.builder.appName("review-enrichment")
    .config("spark.mongodb.read.connection.uri", mongo_uri)
    .config("spark.mongodb.write.connection.uri", mongo_uri)
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/29 15:41:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Structured Streaming: Read Reviews from Kafka

This section reads from the Kafka topic and parses review JSON messages.
Use it to validate streaming ingestion before joining/enriching.


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import from_json, col

KAFKA_BOOTSTRAP = os.getenv('KAFKA_BOOTSTRAP_SERVERS', 'broker:29092')
KAFKA_TOPIC_REVIEW = os.getenv('KAFKA_TOPIC_REVIEW', 'raw_data_review')

review_schema = StructType([
    StructField('review_id', StringType(), True),
    StructField('user_id', StringType(), True),
    StructField('business_id', StringType(), True),
    StructField('stars', IntegerType(), True),
    StructField('useful', IntegerType(), True),
    StructField('funny', IntegerType(), True),
    StructField('cool', IntegerType(), True),
    StructField('text', StringType(), True),
    StructField('date', StringType(), True),
])

kafka_df = (
    spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
    .option('subscribe', KAFKA_TOPIC_REVIEW)
    .option('startingOffsets', 'latest')
    .load()
)

reviews_stream = (
    kafka_df.selectExpr('CAST(value AS STRING) AS json_str')
    .select(from_json(col('json_str'), review_schema).alias('review'))
    .select('review.*')
)

display(reviews_stream)


DataFrame[review_id: string, user_id: string, business_id: string, stars: int, useful: int, funny: int, cool: int, text: string, date: string]

## Start stream (memory sink)

This writes micro-batches into an in-memory table named `reviews_stream`.
Run the query cell below to view results.


In [ ]:
query = (
    reviews_stream.writeStream
    .format('memory')
    .queryName('reviews_stream')
    .outputMode('append')
    .start()
)

query


## Inspect streamed data

Re-run to refresh results as new events arrive.


In [17]:
streamed = spark.table('reviews_stream')
streamed.count()
# streamed.orderBy(streamed['date'].desc()).show(truncate=False)


53

In [10]:
print("Spark:", spark.version)
print("Kafka client:", spark._jvm.org.apache.kafka.common.utils.AppInfoParser.getVersion())


Spark: 4.0.1
Kafka client: 3.9.1


In [18]:
query.stop()

25/12/29 15:47:06 WARN DAGScheduler: Failed to cancel job group f4d11973-3991-4e63-9285-a69db8829bb6. Cannot find active jobs for it.
25/12/29 15:47:06 WARN DAGScheduler: Failed to cancel job group f4d11973-3991-4e63-9285-a69db8829bb6. Cannot find active jobs for it.


In [20]:
for s in spark.streams.active:
    s.stop()
